# CURE-Rec — scalability and second-dataset study

This notebook contains only the two remaining studies: (A) an explicit 6/8/10-player scalability harness and (B) a second chronological external ranking dataset. It never changes the primary six-player YAML configuration and never labels external ranking results as causal policy evidence.

The 8/10-player study uses a declared operational player library; it does not duplicate the original six players. The second dataset must be supplied locally under its license terms.
Run all cells from top to bottom; RUN_ALL is enabled and unavailable/manual-only dataset inputs are recorded as skips rather than fabricated.


In [ ]:
from pathlib import Path
import sys, json, time
from itertools import permutations
from math import factorial
import numpy as np
import pandas as pd

CANDIDATES=[Path.cwd(),Path.cwd()/'paper-ideas'/'CURE-Rec'/'code',*Path.cwd().parents]
ROOT=next(p for p in CANDIDATES if (p/'pyproject.toml').exists() and (p/'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from cure_rec.data import load_dataset
from cure_rec.models import chronological_leave_one_out, PopularityRecommender, BPRMFRecommender, evaluate_user_metrics

RESULTS=ROOT/'results'/'reviewer_phase_assets'
RUN_SCALABILITY=True
RUN_SECOND_DATASET=False
SECOND_DATASET='yahoo_r3'
DOWNLOAD_SECOND_DATASET=False  # Yahoo! R3 is manual-only; the all-run records a skip if unavailable  # explicit consent; Yahoo R3 remains manual-only
SECOND_SOURCE=ROOT/'data'/'raw'/'yahoo-r3'
print('Root:',ROOT)

## A. Distinct-player scalability harness

Declared player library:

6-player core: repeat cap, exploration slot, long-tail slot, diversity reranker, novelty slot, provider balance.

8-player extension: session-length cap, freshness quota.

10-player extension: provider cooldown, category coverage quota.

The extension names are distinct operational transformations and must be implemented in the policy layer before being described as CURE-Rec deployment results. The harness below measures exact-game arithmetic and sampled attribution on a disclosed non-additive benchmark, not causal CURE-Sim policy utility.

In [ ]:
PLAYER_NAMES={6:['repeat_cap','explore_slot','tail_slot','diversify','novel_slot','provider_balance'],8:['repeat_cap','explore_slot','tail_slot','diversify','novel_slot','provider_balance','session_length_cap','freshness_quota'],10:['repeat_cap','explore_slot','tail_slot','diversify','novel_slot','provider_balance','session_length_cap','freshness_quota','provider_cooldown','category_coverage_quota']}

def declared_value(mask,n):
    active=[i for i in range(n) if mask&(1<<i)]
    value=0.02*sum((i+1)**0.5 for i in active)
    if n>=6 and (mask&1) and (mask&2): value+=0.08
    if n>=8 and (mask&(1<<6)) and (mask&(1<<7)): value+=0.04
    if n>=10 and (mask&(1<<8)) and (mask&(1<<9)): value-=0.03
    return value-0.005*len(active)**2

def exact_phi(values,n):
    out=[]
    for i in range(n):
        total=0.0
        others=[j for j in range(n) if j!=i]
        for r in range(n):
            for sub in __import__('itertools').combinations(others,r):
                m=sum(1<<j for j in sub)
                total += factorial(r)*factorial(n-r-1)/factorial(n)*(values[m|(1<<i)]-values[m])
        out.append(total)
    return np.asarray(out)

def sampled_phi(values,n,budget,seed):
    rng=np.random.default_rng(seed); out=np.zeros(n)
    for _ in range(budget):
        order=rng.permutation(n); mask=0
        for i in order:
            nxt=mask|(1<<int(i)); out[int(i)]+=values[nxt]-values[mask]; mask=nxt
    return out/budget

if RUN_SCALABILITY:
    rows=[]
    for n in (6,8,10):
        t=time.perf_counter(); values={m:declared_value(m,n) for m in range(1<<n)}; exact=exact_phi(values,n); exact_time=time.perf_counter()-t
        for budget in (32,128,512,2048):
            t=time.perf_counter(); approx=sampled_phi(values,n,budget,20260815); runtime=time.perf_counter()-t
            rows.append({'players':n,'coalitions':1<<n,'budget':budget,'exact_shapley_seconds':exact_time,'sampled_seconds':runtime,'mae':float(np.mean(np.abs(approx-exact))),'max_error':float(np.max(np.abs(approx-exact))),'sign_agreement':float(np.mean(np.sign(approx)==np.sign(exact))),'efficiency_gap':float(abs(approx.sum()-values[(1<<n)-1])),'player_library':';'.join(PLAYER_NAMES[n])})
    scalability=pd.DataFrame(rows)
    out=RESULTS/'scalability'
    out.mkdir(parents=True,exist_ok=True)
    scalability.to_csv(out/'scalability_exact_vs_sampled.csv',index=False)
    (out/'scalability_manifest.json').write_text(json.dumps({'players':[6,8,10],'budgets':[32,128,512,2048],'scope':'declared non-additive attribution benchmark; not CURE-Sim causal utility','distinct_players':PLAYER_NAMES},indent=2))
    display(scalability)

## Download control

Set `DOWNLOAD_SECOND_DATASET=True` only after checking the dataset terms. The built-in downloader supports datasets whose access terms permit it. Yahoo! R3 remains manual-download-only; for Yahoo! keep it `False` and place the files under `data/raw/yahoo-r3`. The notebook never downloads without explicit consent.


## B. Second external chronological dataset

Set `RUN_SECOND_DATASET=True` only after placing a legally obtained dataset in `SECOND_SOURCE`. Yahoo! R3 is supported as a local loader but is not automatically a long-horizon causal dataset. The loader must pass its audit before ranking results are reported.

In [ ]:
if RUN_SECOND_DATASET:
    try:
        result=load_dataset(SECOND_DATASET,SECOND_SOURCE,download=DOWNLOAD_SECOND_DATASET)
        print(result.metadata)
        print(result.interactions.head())
        split=chronological_leave_one_out(result.interactions)
        popularity=PopularityRecommender().fit(split.train)
        bpr=BPRMFRecommender(max_updates=500_000,seed=42).fit(split.train)
        rows=pd.concat([evaluate_user_metrics(popularity,split,max_users=1000),evaluate_user_metrics(bpr,split,max_users=1000)],ignore_index=True)
        out=RESULTS/'second_dataset'; out.mkdir(parents=True,exist_ok=True)
        rows.to_csv(out/'per_user_metrics.csv',index=False)
        json.dump({'dataset':SECOND_DATASET,'source':str(SECOND_SOURCE),'scope':'chronological warm-item ranking only; not causal policy evidence'},open(out/'manifest.json','w'),indent=2)
        print('Wrote',out/'per_user_metrics.csv')
    else:
    except Exception as exc:
        print(f'Second dataset skipped: {exc}')
        print('Second dataset disabled. Set RUN_SECOND_DATASET=True only after local audit-ready data is available.')


## Required interpretation

The scalability table is an attribution-approximation stress test unless the added players are integrated into CURE-Sim policy semantics. The second-dataset table is descriptive chronological ranking evidence. Neither result establishes long-term causal policy effects from static ratings data.